In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import numpy as np


def plot_random_cached_images_from_shards(train_cache_dir, val_cache_dir, n=5, seed=None):
    """Plot n random images from train/val cyclic cache shard files with labels."""

    def _load_cache(cache_dir):
        cache_dir = Path(cache_dir)
        assert cache_dir.exists(), f"Cache directory not found: {cache_dir}"

        images = []
        labels = []
        shards = sorted(cache_dir.glob('shard_*.pt'))
        for shard in shards:
            data = torch.load(shard, weights_only=True)
            if data['images'].numel() == 0:
                continue
            images.append(data['images'])
            labels.append(data['labels'])

        if len(images) == 0:
            raise RuntimeError(f"No valid images found in cache {cache_dir}")

        images = torch.cat(images, dim=0)
        labels = torch.cat(labels, dim=0)
        return images, labels

    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    train_images, train_labels = _load_cache(train_cache_dir)
    val_images, val_labels = _load_cache(val_cache_dir)

    def _plot_group(imgs, lbls, ax_row):
        idxs = np.random.choice(len(imgs), min(n, len(imgs)), replace=False)
        for col, i in enumerate(idxs):
            img = imgs[i]
            lbl = lbls[i].item() if isinstance(lbls[i], torch.Tensor) else float(lbls[i])
            img_np = img.cpu().numpy() if isinstance(img, torch.Tensor) else np.asarray(img)
            img_np = np.transpose(img_np, (1, 2, 0))
            if img_np.shape[2] == 4:
                img_np = img_np[:, :, :3]
            if img_np.dtype == np.uint8:
                img_np = img_np.astype(np.float32) / 255.0
            elif img_np.max() > 1.0:
                img_np = img_np / img_np.max()

            ax = ax_row[col]
            ax.imshow(img_np)
            ax.axis('off')
            ax.set_title(f"{lbl:.3f}")

    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    fig.suptitle('Random cached images from train (top) and val (bottom)')

    _plot_group(train_images, train_labels, axes[0])
    _plot_group(val_images, val_labels, axes[1])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig("cache_inspection.png", dpi=450)


# Example usage:
plot_random_cached_images_from_shards(
    train_cache_dir='/home/abbatenicolas/data/cache/train_cache',
    val_cache_dir='/home/abbatenicolas/data/cache/val_cache',
    n=5,
    seed=42
)